# Preclass
AI for Materials Science — Hands-on session 1

This is a warm-up for the tools you will use in class.
We will work through NumPy for numeric arrays, pandas for tables, and pymatgen for compositions and
crystal structures.

## What this notebook covers

### 0 · Setup
Install the library and download the course data.

### 1 · NumPy
Build an array and pick out only the values that meet a condition.

### 2 · pandas
Read a CSV, check for missing values, and shortlist candidate steels.

### 3 · pymatgen
Parse a chemical formula, and read and write crystal structures as CIF files.

---

Run the cells one at a time from the top.
Later cells reuse variables created in earlier ones, so skipping ahead gives you a "name is not
defined" error.
Lines starting with `##` inside the code are comments written for you; Python does not run them.

## 0. Setup

Two things to get in place first: the library that runs the code, and the data we will read.

### 0-1. Install the library
`pymatgen` is the only package you need to install.
NumPy and pandas come along with it, because pymatgen uses them internally.
Feel free to read ahead while the install runs.

In [ ]:
## A leading ! runs a terminal command instead of Python. -q keeps the install output quiet.
!pip install -q pymatgen

### 0-2. Download the course material
The CSV and CIF files for this session live in the `Data/` folder of the course repository.
Running the cell below copies the whole repository into your current working directory.
You do not need to type a path from your own computer or mount Google Drive.

If you run it a second time you will see a "destination path already exists" message.
That is harmless, so just move on to the next cell.

In [ ]:
## git clone downloads an entire repository from GitHub.
!git clone https://github.com/kwongibaek/MS49900-AI4M.git

### 0-3. Import the libraries
Installing and importing are two different steps.
Installing puts files somewhere on the machine; `import` brings a tool into *this* notebook.
So even after a successful install, skipping the import still gives you a "name is not defined" error.

`as` attaches a short alias. Instead of typing `numpy` every time, we write `np`.

In [ ]:
## Standard Python tool for file paths.
from pathlib import Path

## np for numeric arrays, pd for tables.
import numpy as np
import pandas as pd

### 0-4. Set the file paths
We decide once where the files live, and from here on we refer to them by short names like
`STEEL_CSV`.
That saves retyping file names, and if a location ever changes you only edit this one cell.

Running the cell prints the list of files inside `Data/`.
If you get an error instead of a list, check that you ran the `git clone` cell in 0-2.

In [ ]:
## The Data folder inside the repository downloaded in 0-2.
DATA_DIR = Path("MS49900-AI4M/Data")

## / joins a folder name and a file name into one path.
STEEL_CSV = DATA_DIR / "steel_strength.csv"
LFP_CIF = DATA_DIR / "LiFePO4.cif"

## Anything we produce in this session is saved here. The folder is created if it does not exist.
OUTPUT_DIR = Path("outputs/01_preclass")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

sorted(path.name for path in DATA_DIR.iterdir())

## 1. NumPy: picking values out of an array

Materials data is, in the end, a lot of numbers. The tool for handling those bundles of numbers is
the NumPy **array**.

It looks a lot like a Python list, but the difference is that you can compute on the whole bundle at
once instead of looping over values one by one. The bigger the dataset, the more that difference
matters.

Here we will build an array from five yield strength values and practise selecting the ones we want.
Real experimental data comes in the next section.

### 1-1. Creating an array
Pass a list of numbers to `np.array()` and you get an array.

In the output, `dtype` tells you how the values are stored. These have decimal points, so we get
`float64`.
`shape` is the size of the array: `(5,)` means five values laid out in a single row.

In [ ]:
## np is the NumPy alias we set up in 0-3. The array built on the right is stored under the name on the left.
strength = np.array([620.0, 780.0, 910.0, 1050.0, 1180.0])

print("Array:", strength)
print("dtype and shape:", strength.dtype, strength.shape)

### 1-2. Pulling out values by position
To take a single value out of an array, put its position in square brackets.
Watch out for one thing: Python counts positions **from 0**. The first value is `[0]`, not `[1]`.

A negative index like `[-1]` counts from the end, which is handy for the last value.
A colon, as in `[1:4]`, selects a range. The start is included and the end is not, so you get the
three values at positions 1, 2 and 3.

In [ ]:
print("First value:", strength[0])
print("Last value:", strength[-1])
print("A slice:", strength[1:4])

### 1-3. Applying the same operation to a whole array
Add a single number to an array and it is added to every value at once. No loop needed.
This is how you would correct measurements or convert units.

By the way, if the last line of a cell is just a variable name, its value is shown without `print()`.

In [ ]:
## strength itself is left alone; the shifted values go into a new array called adjusted.
adjusted = strength + 25.0
adjusted

### 1-4. Selecting by condition
Apply a comparison like `>=` to an array and the result is not a single True/False but a
**True/False array with one entry per value**.
This is called a mask.

Most of the time, selecting the values you want goes through a mask like this.
Let us look at what the mask itself looks like first.

In [ ]:
## Each of the five values is compared against 900.
mask = strength >= 900.0
mask

Now we use that mask to pull out the actual values.
Putting the mask in square brackets keeps only the values where it is True, and you can compute a
count or a mean directly on what is left.

In [ ]:
selected = strength[mask]

## size is how many values there are, mean() is their average.
print("Selected values:", selected)
print("Count and mean:", selected.size, selected.mean())

## 2. pandas: reading a CSV and shortlisting steels

Now for some real experimental data.
`steel_strength.csv` holds the composition and measured properties of 312 steels.
Element contents are in wt%, yield and tensile strength in MPa, and elongation in %.

A table with rows and columns like this is called a **DataFrame** in pandas.
It helps to think of it as an Excel sheet you can work with from Python.

Our goal is to filter for steels that are "strong and still ductile".
We will go read → inspect → filter → save.

### 2-1. Reading the CSV
Give `pd.read_csv()` a file path and it reads the table in.
`head()` shows only the first five rows; it is a good habit to check the column names and the shape
of the values the moment you open a dataset.

In [ ]:
steel = pd.read_csv(STEEL_CSV)
steel.head()

Let us also check how big the table is.
`shape` gives (number of rows, number of columns) at once, while `len()` gives just the row count.

In [ ]:
print("Rows and columns:", steel.shape)
print("Total rows:", len(steel))

### 2-2. Selecting just the columns you need
Put one column name in square brackets to get that column, or a list of names to get several at once.
There are 17 columns here, so let us set aside just the three properties we care about.

In [ ]:
## Three column names bundled into a list and selected in one go.
property_columns = ["yield strength", "tensile strength", "elongation"]
properties = steel[property_columns]
properties.head()

### 2-3. Checking for missing values
Experimental datasets almost always have gaps where a measurement was not taken. pandas marks those
blanks as `NaN`.

Before applying any condition, let us count how many blanks each column has.
Note that only the elongation column has any. That is exactly what forces an extra step in the next
part.

In [ ]:
## count() counts filled cells; isna().sum() counts empty ones, column by column.
pd.DataFrame({
    "valid_values": properties.count(),
    "missing_values": properties.isna().sum(),
})

### 2-4. Building two conditions
We want "yield strength at least 1500 MPa" and "elongation at least 10%", each written as its own
condition.
The mechanics are identical to the mask in 1-4: each condition becomes a list of True/False, one per
row.

The elongation condition also has `notna()` in it.
Blanks would compare as False anyway, but spelling out "the value exists *and* it is at least 10"
makes the intent obvious when you reread the code later.

In [ ]:
## Each condition is a True/False list covering all 312 rows.
strength_condition = steel["yield strength"] >= 1500.0
elongation_condition = steel["elongation"].notna() & (steel["elongation"] >= 10.0)

## Summing True as 1 tells you how many rows pass each condition.
print("Passing the strength condition:", int(strength_condition.sum()))
print("Passing the elongation condition:", int(elongation_condition.sum()))

### 2-5. Keeping rows that satisfy both
Joining two conditions with `&` keeps only the rows where both are True.
Notice how much smaller that is than either condition alone; stacking conditions narrows a candidate
list fast.

`loc[condition]` is the part that actually selects the rows, and `sort_values()` orders them by
descending strength.

In [ ]:
## copy() makes a table separate from the original steel, so later edits do not affect it.
candidates = steel.loc[strength_condition & elongation_condition].copy()
candidates = candidates.sort_values("yield strength", ascending=False)

print("Passing both conditions:", len(candidates))

## A table on the last line of a cell is displayed as the cell result.
candidates[["formula", *property_columns]].head()

### 2-6. Saving the result
Writing the shortlist to a file means you can reuse it in the next session or in a report.
The CSV goes into the `outputs/01_preclass/` folder.

In [ ]:
## index=False keeps the row numbers pandas added out of the saved CSV.
candidate_csv = OUTPUT_DIR / "steel_candidates.csv"
candidates.to_csv(candidate_csv, index=False)

print("Saved file:", candidate_csv)

## 3. pymatgen: chemical formulas and crystal structures

So far we have looked at materials only as tables of numbers. Now for a tool that handles the
materials themselves.

pymatgen has two objects with different scope.

- `Composition` covers only the composition: what is in it and how much.
- `Structure` adds the lattice and the atomic positions, so it carries the full crystal structure.

We will parse the LiFePO₄ formula, build a silicon crystal structure from scratch and save it as a
CIF file, then read a prepared LiFePO₄ CIF back in.

As an aside, when looking for experimentally determined crystal structures, ICSD normally requires a
licence while COD is openly accessible.
The CIF we read here is an example file published with pymatgen.

### 3-1. Parsing a chemical formula
Pass a formula as a string, as in `Composition("LiFePO4")`, and pymatgen works out the elements and
their counts on its own.
It converts human notation into something a program can compute with.

In [ ]:
## Three things from pymatgen for this section.
from pymatgen.core import Composition, Lattice, Structure

lfp_composition = Composition("LiFePO4")

## reduced_formula is the formula in the simplest integer ratio; num_atoms is atoms per formula unit.
print("Reduced formula:", lfp_composition.reduced_formula)
print("Atoms per formula unit:", lfp_composition.num_atoms)

The atomic fraction of an element is its atom count divided by the total number of atoms.
LiFePO₄ has 4 oxygen atoms out of 7, so oxygen comes out at 4/7 ≈ 0.571.

In [ ]:
lfp_composition.fractional_composition.as_dict()

### 3-2. Building a structure from symmetry
You do not have to list every atomic position to build a crystal structure.
Once the space group is fixed, the remaining sites follow by symmetry from a single representative
atom.

Silicon has the diamond structure (space group Fd-3m) with a lattice constant of 5.431 Å.
Check that a single atomic position fills the unit cell with 8 sites.

In [ ]:
## species and coords hold one representative atom; Fd-3m symmetry generates the rest.
silicon = Structure.from_spacegroup(
    "Fd-3m", Lattice.cubic(5.431), species=["Si"], coords=[[0, 0, 0]],
)

## abc holds the three lattice edge lengths, in Å.
print("Lattice lengths:", silicon.lattice.abc)
print("Sites in this cell:", len(silicon))

### 3-3. Saving as a CIF file
CIF is the standard format for exchanging crystal structures.
Saved to a file, the structure opens in visualisation programs like VESTA and in calculation codes.

In [ ]:
## fmt="cif" chooses the output format.
silicon_cif = OUTPUT_DIR / "silicon.cif"
silicon.to(filename=str(silicon_cif), fmt="cif")

print("Saved structure:", silicon_cif)

### 3-4. Reading a CIF file
If you can write one, you can read one. This time we read the prepared LiFePO₄ CIF.

In 3-1 we only had the formula; here we also get the lattice size and the number of atoms.
There are 28 sites because this file's unit cell contains four formula units (7 × 4 = 28).

In [ ]:
## from_file() reads a file and turns it into a Structure object.
lfp_structure = Structure.from_file(LFP_CIF)

print("Formula:", lfp_structure.composition.reduced_formula)
print("Lattice lengths:", lfp_structure.lattice.abc)
print("Number of sites:", len(lfp_structure))

You can also reach individual atoms inside a structure. `[0]` is the first site.

The **fractional coordinates** printed here are not distances in Å. They are positions measured with
each unit cell edge taken as 1.
A coordinate (x, y, z) means "x of the way along the a axis, y along b, z along c".
The a axis of this CIF is about 10.4 Å, so an atom at x = 0.5 sits roughly 5.2 Å along a.

There are two reasons to use this instead of Å.
Values fall between 0 and 1, so you can read off at a glance that (0.5, 0.5, 0.5) is the centre of
the cell.
And the coordinates stay the same when the lattice size changes, as long as the structure does.
That holds when a cell expands with temperature or pressure, and across same-structure materials of
different size such as Si and Ge.
It is also why we got 8 sites from a single atomic position in 3-2: symmetry operations are simple in
this coordinate system.

If you see a value close to 1 such as 0.99999 in the output, read it as 0.
The cell repeats infinitely, so adding 1 to a coordinate lands on the same site in the neighbouring
cell. In other words, this atom sits at a corner of the cell.

In [ ]:
print("First element:", lfp_structure[0].species_string)
print("Fractional coordinates of the first atom:", lfp_structure[0].frac_coords)

## Data sources

### Steel strength data

`Data/steel_strength.csv` contains the compositions and measured properties of 312 steels.
These experimental data were obtained from **Mechanical properties of some steels**, published
on the Citrination materials data platform. The dataset was cleaned and duplicate records were
removed for distribution through matminer.

- **Original dataset:** Citrine Informatics, [Mechanical properties of some steels (Citrination dataset 153092)](https://citrination.com/datasets/153092/).
- **Dataset used in this course:** Hacking Materials, [Steel Strength Data](https://doi.org/10.6084/m9.figshare.7250453). Available in matminer as `steel_strength`.
- **Column descriptions and units:** [matminer dataset documentation](https://hackingmaterials.lbl.gov/matminer/dataset_summary.html#steel-strength).
